Train Markov Model

In [59]:
import numpy as np

new_txt = "one fish two fish red fish blue fish"

def build_markov_model(markov_model, new_txt, order =1):
    
    if isinstance(new_txt, str):
        words = new_txt.lower().split()
    else:
        words = new_txt

    #insert star and end

    start_token = ['*S*'] * order
    end_token = ['*E*']

    words = start_token + words + end_token

    #iterate though words
    for i in range(len(words) - order):

        # get current state
        current_state = tuple(words[i:i + order])
        
        # get next word
        next_word = words[i + order]

        #update markov model
        # if current state not in markov model
        if current_state not in markov_model:
            markov_model[current_state] = {}

        #if next word not in dic
        if next_word not in markov_model[current_state]:
            markov_model[current_state][next_word] = 0

        # count all frequencies
        markov_model[current_state][next_word] += 1

    return markov_model







Gernate next word from modle

In [60]:
import numpy as np

def get_next_word(current_word, markov_model, seed= 42):
    
    if seed is not None:
        np.random.seed(seed)

    #check current_word if in the markov_model
    if current_word in markov_model:
        
        # get next_words (dic)
        
        next_words_dict = markov_model[current_word]
        
        # exact the key(next_word) values(frenqucy)
        candidates = list(next_words_dict.keys())
        counts = list(next_words_dict.values())
        
        # caculate total_count
        total_count = sum(counts)
        
        # caculate probobilitys for each word
        # P(word) = count(word) / total_count
        probabilities = [count / total_count for count in counts]
        
        # np.random.choice to choosen next word randomly based on probobilitys
        
        chosen_word = np.random.choice(candidates, p=probabilities)
        
        # return chosen word
        return chosen_word
        
    else:
        # cuurent_word not in markov_mopdel
        return None

In [118]:
import numpy as np

new_txt = "one fish two fish red fish blue fish"
def generate_random_text(markov_model, seed=42):
    """
    generate text depend on markov_modle
    """
    # check order
    # keys [('one', 'fish'), ('fish', 'two')] -> order = 2
    if not markov_model:
        return ""

    first_key = list(markov_model.keys())[0]
    order = len(first_key)

    # set random seed
    if seed is not None:
        np.random.seed(seed)

    # 2. set current_state
    current_state = tuple(["*S*"] * order)

    # 3. set a sentence
    sentence = []

    # 4. generate next_word and combine
    while True:
        # use the get_next_word we made
        next_word = get_next_word(current_state, markov_model, seed=None)

        # check the end and None
        if next_word == "*E*" or next_word is None:
            break

        # c.  append next_word to sentence
        sentence.append(next_word)

        # Sliding Window to remove old variable and add the new
        # transform next word to tuple
        current_state = current_state[1:] + (next_word,)

    # 5. combine the sentence with ""
    return " ".join(sentence)

Gerate new_txt

In [ ]:
def gen_sentence(mode="line", path="one_fish_two_fish", 
                 order=1, seed=None):
    
    def split_sentences(text_content):
        text_content = text_content.replace("\n", " ") 
        split_text = [[]]
        text_content = text_content.split(" ")
        for i in range(len(text_content)):
            if len(text_content[i]) > 0:
                split_text[0].append(text_content[i])
                if text_content[i][0] in {"!", "?", ".", '"'} and len(text_content[i]) == 1:
                    if i < len(text_content) - 1:
                        if text_content[i + 1] != '"':
                            split_text.insert(0, [])
        return split_text
    
    def format_output(text):
        text = text.split(" ")
        for i in range(len(text) - 1, -1, -1):
            if text[i] in {"*S*", "*E*"}:
                text.pop(i)
            elif len(text[i]) == 1:
                if ord(text[i]) in range(33, 64):
                    text[i - 1] += text[i]
                    if i < len(text) - 1:
                        if text[i] in {"!", ".", "?"}:
                            text[i + 1] = text[i + 1][0].capitalize() + text[i + 1][1:]
                    text.pop(i)
                elif text[i] == "i":
                    text[i] = text[i].capitalize()
        output = " ".join(text)
        output = output[0].capitalize() + output[1:]
        return output


    if mode not in {"line", "sentence", "sonnet"}:
        raise ValueError("WRONG MODE")

    try:
        with open(f"data/{path}.txt", "r") as f:        # open file
            text_content = f.read()                     # 2. read all line
    except FileNotFoundError:
        print(f"error: can't find the file {path} exist?")


    for p in {",", ".", "!", "?", "\"", ";", ":"}:              # define punctuation
        text_content = text_content.replace(p, " " + p)         #  "," transform to " , " for split in markov_modle
    text_content = text_content.lower()


    if mode == "sentence":
        text_content = split_sentences(text_content)
    elif mode == "line":
        text_content = text_content.split("\n")
    elif mode == "sonnet":
        text_content = text_content.replace("\n", " \n ").split("  ")
    text_content = [entry for entry in text_content if len(entry) > 0]

    markov_model = {}
    for entry in text_content:
        markov_model = build_markov_model(markov_model, entry, order=order)

    output = generate_random_text(markov_model, seed=seed)      # spawn output
    
    formatted_output = format_output(output)
    print(formatted_output)
    return formatted_output

x = gen_sentence(mode="sonnet", path="sonnets", order=1)

Let me behold these contents than of the work did impute, whose blessed made by chance, not figur'd to mortal moon, as stone, and I can speak of fortune chide: he have I lie.


Gerate shakespeare style sonnet

In [63]:
import numpy as np


# new dic
sonet_markov_model = dict()
sonet = "" # save t temporary

try:
    with open("data/sonnets.txt", "r") as f:
        # iterate lines
        for line in f:
            # strip
            line = line.strip()
            
            # to check""(the end fo a sonet)
            if line == "":
                # when sonet not empty
                if len(sonet) > 0:
                
                    # punctuation deine
                    for p in [",", ".", "!", "?", ":", ";", "'"]:
                        sonet = sonet.replace(p, " " + p + " ")
                    
                    # train modle with one of sonet
                    sonet_markov_model = build_markov_model(sonet_markov_model, sonet, order=42)
                    
                    # empty sonet for next sonet
                    sonet = ""
            
            else: 
                # cumulate
                
                sonet = sonet + " " + line

    # when loop end handle the last sonet
    if len(sonet) > 0:
        # punttuation
        for p in [",", ".", "!", "?", ":", ";", "'"]:
            sonet = sonet.replace(p, " " + p + " ")
            
        sonet_markov_model = build_markov_model(sonet_markov_model, sonet, order=2)

    # generate shakespeare style
    print("sharke style")
    
    print(generate_random_text(sonet_markov_model, seed=42))

except FileNotFoundError:
    print("erro : can't findsonnets.txt")

sharke style
so , now i have confess ' d that he is thine , and i my self am mortgag ' d to thy will , myself i ' ll forfeit , so that other mine thou wilt restore to be my comfort still : but thou wilt not , nor he will not be free , for thou art covetous , and he is kind ; he learn ' d but surety-like to write for me , under that bond that him as fast doth bind . the statute of thy beauty thou wilt take , thou usurer , that putt ' st forth all to use , and sue a friend came debtor for my sake ; so him i lose through my unkind abuse . him have i lost ; thou hast both him and me : he pays the whole , and yet am i not free .
